###**Loading the CSV Files generated from Mockaroo**:

In [64]:
!curl "https://api.mockaroo.com/api/9de85370?count=1000&key=84868b20" > "Employee.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  186k    0  186k    0     0  49081      0 --:--:--  0:00:03 --:--:-- 49079


In [65]:
!curl "https://api.mockaroo.com/api/c59cee30?count=1000&key=84868b20" > "Employee-Country.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  185k    0  185k    0     0  83880      0 --:--:--  0:00:02 --:--:-- 83850


In [66]:
!curl "https://api.mockaroo.com/api/0c4359b0?count=1000&key=84868b20" > "Employee-Gender.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  151k    0  151k    0     0  54520      0 --:--:--  0:00:02 --:--:-- 54563


In [ ]:
!curl "https://api.mockaroo.com/api/bf51f230?count=1000&key=84868b20" > "Employee-Role.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  210k    0  210k    0     0  70613      0 --:--:--  0:00:03 --:--:-- 70689


###**Liberies**:

In [102]:
import sqlite3
import pandas as pd
import os

##**Creating Tables**:

In [88]:
employee_role_schema = """
create table Employee_Role (
        employee_id INT,
        first_name VARCHAR(50),
        last_name VARCHAR(50),
        gender VARCHAR(50),
        date_of_birth DATE,
        job_title VARCHAR(50),
        department VARCHAR(50),
        salary DECIMAL(8,2),
        hire_date DATE,
        email VARCHAR(50),
        phone_number VARCHAR(50),
        address VARCHAR(50),
        city VARCHAR(50),
        state VARCHAR(50),
        postal_code VARCHAR(50),
        country VARCHAR(50),
        emergency_contact_name VARCHAR(50),
        emergency_contact_phone VARCHAR(50),
        emergency_contact_relationship VARCHAR(7),
        manager_id INT,
        start_date DATE
);
"""
employee_country_schema = """
create table Employee_Country (
        employee_id INT,
        first_name VARCHAR(50),
        last_name VARCHAR(50),
        gender VARCHAR(10),
        age INT,
        email VARCHAR(50),
        country VARCHAR(50),
        postal_code VARCHAR(50),
        department VARCHAR(50),
        salary DECIMAL(8,2),
        hire_date DATE,
        manager_id INT,
        performance_rating DECIMAL(2,1),
        job_title VARCHAR(50),
        work_location VARCHAR(50),
        work_schedule VARCHAR(50),
        vacation_days INT,
        start_time VARCHAR(50),
        end_time VARCHAR(50),
        overtime_hours INT,
        years_of_service INT,
        education_level VARCHAR(50),
        language_spoken VARCHAR(50),
        ethnicity VARCHAR(50),
        benefits_package VARCHAR(50)
);
"""
employee_gender_schema = """
create table Employee_Gender (
        employee_id INT,
        first_name VARCHAR(50),
        last_name VARCHAR(50),
        gender VARCHAR(10),
        age INT,
        email VARCHAR(50),
        country VARCHAR(50),
        postal_code VARCHAR(50),
        department VARCHAR(9),
        salary DECIMAL(8,2),
        hire_date DATE,
        manager_id INT,
        performance_rating DECIMAL(2,1),
        job_title VARCHAR(50),
        work_location VARCHAR(6),
        work_schedule VARCHAR(9),
        vacation_days INT,
        start_time VARCHAR(50),
        end_time VARCHAR(50),
        overtime_hours INT
);
"""
employee_schema = """
create table Employee (
        employee_id INT,
        first_name VARCHAR(50),
        last_name VARCHAR(50),
        age INT,
        email VARCHAR(50),
        gender VARCHAR(4),
        job_title VARCHAR(4),
        department VARCHAR(50),
        salary DECIMAL(8,2),
        hire_date DATE,
        phone_number VARCHAR(50),
        address VARCHAR(50),
        city VARCHAR(50),
        state VARCHAR(50),
        postal_code VARCHAR(50),
        country VARCHAR(4),
        emergency_contact_name VARCHAR(50),
        emergency_contact_phone VARCHAR(50),
        emergency_contact_relationship VARCHAR(7),
        manager_id INT,
        start_date DATE,
  FOREIGN KEY (gender) REFERENCES customers(gender),
  FOREIGN KEY (country) REFERENCES products(country),
  FOREIGN KEY (department) REFERENCES products(department),
  FOREIGN KEY (job_title) REFERENCES products(job_title)
);
"""

# **Remove Existing Database**

This small Python snippet checks if a database file named `employee.db` exists. If it does, the file is **deleted**, effectively removing the existing database. This is useful when you want to start fresh and recreate the database from scratch.

**Explanation:**

* `os.path.exists(db_name)` → Checks if the database file exists.
* `os.remove(db_name)` → Deletes the file.
This ensures that any old tables or data do not interfere with creating a new database.


In [103]:
db_name = 'employee.db'
if os.path.exists(db_name):
  os.remove(db_name)
  print(f"Existing database '{db_name}' removed")

Existing database 'employee.db' removed


# **Load Employee Data into SQLite Database**

This Python script automates the process of creating and populating an `employee.db` SQLite database from CSV files.

**What it does:**

1. **Defines Column Types:**
   A dictionary (`COLUMN_DATA_TYPES`) maps each table to its expected column names and data types. This ensures data from CSV files is correctly interpreted.

2. **Connects to SQLite:**
   Opens a connection to `employee.db`. If the database does not exist, it will be created.

3. **Maps CSVs to Tables:**
   Each CSV file (`Employee.csv`, `Employee-Role.csv`, etc.) is associated with its corresponding SQLite table.

4. **Drops Existing Tables:**
   Any pre-existing tables with the same name are dropped to avoid conflicts with the new schema.

5. **Creates Tables:**
   Uses pre-defined table schemas (`employee_role_schema`, `employee_country_schema`, etc.) to create the tables in SQLite.

6. **Loads Data from CSVs:**
   For each CSV file that exists, the data is read with pandas and appended to the corresponding table in the database. Missing CSV files are skipped with a warning.

7. **Commits and Closes:**
   All changes are committed to the database, and the connection is closed safely.

**Outcome:**
After running this script, the `employee.db` database contains fully populated tables (`Employee`, `Employee_Role`, `Employee_Country`, `Employee_Gender`) ready for querying.


In [104]:
import os, sqlite3, pandas as pd

COLUMN_DATA_TYPES = {
    'employee_role': {
        'employee_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'email': 'object',
        'phone_number': 'object',
        'address': 'object',
        'city': 'object',
        'country': 'object',
        'postal_code': 'object',
        'role_title': 'object',
        'salary_grade': 'object',
        'hire_date': 'datetime64[ns]'
    },

    'employee_country': {
        'employee_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'gender': 'object',
        'age': 'int64',
        'email': 'object',
        'country': 'object',
        'postal_code': 'object',
        'department': 'object',
        'salary': 'float64',
        'hire_date': 'datetime64[ns]',
        'manager_id': 'int64',
        'performance_rating': 'float64',
        'job_title': 'object',
        'work_location': 'object',
        'work_schedule': 'object',
        'vacation_days': 'int64',
        'start_time': 'object',
        'end_time': 'object',
        'overtime_hours': 'int64',
        'years_of_service': 'int64',
        'education_level': 'object',
        'language_spoken': 'object',
        'ethnicity': 'object',
        'benefits_package': 'object'
    },

    'employee_gender': {
        'employee_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'gender': 'object',
        'age': 'int64',
        'email': 'object',
        'country': 'object',
        'postal_code': 'object',
        'department': 'object',
        'salary': 'float64',
        'hire_date': 'datetime64[ns]',
        'manager_id': 'int64',
        'performance_rating': 'float64',
        'job_title': 'object',
        'work_location': 'object',
        'work_schedule': 'object',
        'vacation_days': 'int64',
        'start_time': 'object',
        'end_time': 'object',
        'overtime_hours': 'int64'
    },

    'employee': {
        'employee_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'age': 'int64',
        'email': 'object',
        'gender': 'object',
        'job_title': 'object',
        'department': 'object',
        'salary': 'float64',
        'hire_date': 'datetime64[ns]',
        'phone_number': 'object',
        'address': 'object',
        'city': 'object',
        'state': 'object',
        'postal_code': 'object',
        'country': 'object',
        'emergency_contact_name': 'object',
        'emergency_contact_phone': 'object',
        'emergency_contact_relationship': 'object',
        'manager_id': 'int64',
        'start_date': 'datetime64[ns]'
    }
}

db_name = 'employee.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

csv_to_table = {
    '/content/Employee.csv': 'Employee',
    '/content/Employee-Role.csv': 'Employee_Role',
    '/content/Employee-Country.csv': 'Employee_Country',
    '/content/Employee-Gender.csv': 'Employee_Gender'
}

# Drop tables to avoid schema conflicts on retry
for table_name in csv_to_table.values():
    cursor.execute(f"DROP TABLE IF EXISTS {table_name};")

# Create tables with corrected schemas
cursor.execute(employee_role_schema)
cursor.execute(employee_country_schema)
cursor.execute(employee_gender_schema)
cursor.execute(employee_schema)


for csv_file, table_name in csv_to_table.items():
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        df.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Loaded {len(df)} rows into {table_name}")
    else:
        print(f"CSV not found: {csv_file}")

conn.commit()
conn.close()

Loaded 1000 rows into Employee
Loaded 1000 rows into Employee_Role
Loaded 1000 rows into Employee_Country
Loaded 1000 rows into Employee_Gender


# **Setup Google GenAI Client**

This snippet installs the **Google GenAI Python library** and initializes a client using an API key stored in Colab.

**Steps explained:**

1. **Install the library**
   `!pip install --quiet google-genai` installs the package quietly without verbose output.

2. **Import the library**
   `from google import genai` imports the GenAI client module.

3. **Access stored API key**
   `from google.colab import userdata` allows you to retrieve secrets stored in your Colab environment.
   `userdata.get('GOOGLE_API_KEY')` fetches your Google API key.

4. **Initialize the client**
   `genai_client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))` creates a GenAI client that can now interact with Google’s Generative AI services, such as text-to-SQL or text generation.

**Outcome:**
After running this, `genai_client` is ready to send prompts and receive responses from Google’s GenAI models directly in Colab.


In [ ]:
!pip install --quiet google-genai

In [71]:
from google import genai
from google.colab import userdata

In [72]:
genai_client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

###**This prompt is designed for a Natural Language to SQL (NL2SQL) engine that converts plain English questions about employees into executable SQLite queries.**
To use the templet, you may checkout:
https://www.geeksforgeeks.org/data-science/a-unified-framework-for-an-effective-prompt/

In [73]:
employee_prompt = """
### **ROLE**
You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL
(NL2SQL) translation. Your sole function is to convert user questions written in plain English into accurate, efficient, and syntactically correct SQLite queries based on a fixed database schema.
-----
### **CONTEXT**
You are the core translation engine for a business intelligence dashboard. This tool allows non-technical employees to query the company's employee database using natural language. The database dialect is always **SQLite**. Your responses will be executed directly on the database.
The database consists of the following four tables:

**`Employee-Role` table:**
```sql
CREATE TABLE Employee-Role (
    employee_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    gender VARCHAR(50),
    date_of_birth DATE,
    job_title VARCHAR(50),
    department VARCHAR(50),
    salary DECIMAL(8,2),
    hire_date DATE,
    email VARCHAR(50),
    phone_number VARCHAR(50),
    address VARCHAR(50),
    city VARCHAR(50),
    state VARCHAR(50),
    postal_code VARCHAR(50),
    country VARCHAR(50),
    emergency_contact_name VARCHAR(50),
    emergency_contact_phone VARCHAR(50),
    emergency_contact_relationship VARCHAR(7),
    manager_id INT,
    start_date DATE
);
````

**`Employee-Country` table:**

```sql
CREATE TABLE Employee-Country (
    employee_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    gender VARCHAR(10),
    age INT,
    email VARCHAR(50),
    country VARCHAR(50),
    postal_code VARCHAR(50),
    department VARCHAR(9),
    salary DECIMAL(8,2),
    hire_date DATE,
    manager_id INT,
    performance_rating DECIMAL(2,1),
    job_title VARCHAR(50),
    work_location VARCHAR(6),
    work_schedule VARCHAR(9),
    vacation_days INT,
    start_time VARCHAR(50),
    end_time VARCHAR(50),
    overtime_hours INT
);
```

**`Employee-Gender` table:**

```sql
CREATE TABLE Employee-Gender (
    employee_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    gender VARCHAR(10),
    age INT,
    email VARCHAR(50),
    country VARCHAR(50),
    postal_code VARCHAR(50),
    department VARCHAR(9),
    salary DECIMAL(8,2),
    hire_date DATE,
    manager_id INT,
    performance_rating DECIMAL(2,1),
    job_title VARCHAR(50),
    work_location VARCHAR(6),
    work_schedule VARCHAR(9),
    vacation_days INT,
    start_time VARCHAR(50),
    end_time VARCHAR(50),
    overtime_hours INT
);
```

**`Employee` table:**

```sql
CREATE TABLE Employee (
    employee_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    age INT,
    email VARCHAR(50),
    gender VARCHAR(4),
    job_title VARCHAR(4),
    department VARCHAR(50),
    salary DECIMAL(8,2),
    hire_date DATE,
    phone_number VARCHAR(50),
    address VARCHAR(50),
    city VARCHAR(50),
    state VARCHAR(50),
    postal_code VARCHAR(50),
    country VARCHAR(4),
    emergency_contact_name VARCHAR(50),
    emergency_contact_phone VARCHAR(50),
    emergency_contact_relationship VARCHAR(7),
    manager_id INT,
    start_date DATE,
    FOREIGN KEY (gender) REFERENCES customers(gender),
    FOREIGN KEY (country) REFERENCES products(country),
    FOREIGN KEY (department) REFERENCES products(department),
    FOREIGN KEY (job_title) REFERENCES products(job_title)
);
```

---

### **TASK**

Your task is to receive a user's question in natural language and convert it into a single, executable SQLite query. Follow these steps meticulously:

1. **Analyze the User's Query:** Deconstruct the user's question to understand the core intent. Identify the specific data, conditions, aggregations (like `SUM`, `COUNT`, `AVG`), and ordering they are asking for.
2. **Map to the Schema:** Map the entities from the user's query to the appropriate table(s) (`Employee`, `Employee-Role`, `Employee-Country`, `Employee-Gender`) and columns. Determine necessary `JOIN` operations based on foreign keys.
3. **Construct the SQLite Query:** Write a clean and efficient `SELECT` statement that is syntactically correct for SQLite. Ensure all table and column names are accurate.
4. **Handle Ambiguity:** If the user's query is vague, ambiguous, or lacks necessary information, do not guess. Instead, formulate a specific clarifying question.

---

### **CONSTRAINTS**

* **Read-Only Operations:** Only generate `SELECT` queries. Never generate `INSERT`, `UPDATE`, `DELETE`, `DROP`, or any other modifying statements.
* **Adhere Strictly to Schema:** Use only the tables and columns defined above. Do not invent or assume other tables/columns.
* **No Explanations:** Do not add conversational text or explanations about the query.
* **Single Query Only:** The output must be a single, complete SQLite query.
* **Handle Impossibility:** If the request cannot be fulfilled with the given schema, clearly state why.

---

### **OUTPUT FORMAT**

Your final response must be a single JSON object with two keys:

1. `"status"`: `"success"`, `"clarification_needed"`, or `"error"`.
2. `"response"`:

   * If `"success"`, include the complete SQLite query.
   * If `"clarification_needed"`, include a clarifying question.
   * If `"error"`, include a reason why the query cannot be generated.

---

### **EXAMPLES**

**Example 1: Simple Lookup**

* **User Query:** "Show all employees in the Sales department"

```json
{
"status": "success",
"response": "SELECT * FROM Employee WHERE department = 'Sales';"
}
```

**Example 2: Complex Join**

* **User Query:** "Show employees with their country and role"

```json
{
"status": "success",
"response": "SELECT e.first_name, e.last_name, c.country, r.job_title FROM Employee AS e INNER JOIN Employee-Country AS c ON e.employee_id = c.employee_id INNER JOIN Employee-Role AS r ON e.employee_id = r.employee_id;"
}
```

**Example 3: Ambiguous Query**

* **User Query:** "Show recent hires"

```json
{
"status": "clarification_needed",
"response": "Please specify what 'recent' means (e.g., last 7 days, this month)."
}
```

**Example 4: Impossible Query**

* **User Query:** "Which warehouse has the most stock?"

```json
{
"status": "error",
"response": "I cannot answer this question as the database does not contain warehouse information."
}
```

"""


## **Generate SQL Query from Natural Language**:

This function takes a user’s natural language query and converts it into an **executable SQLite query** using Google’s GenAI model.

**Explanation:**

* `genai_client.models.generate_content` → Sends the prompt and user query to the model.
* `usage_metadata` → Tracks token usage for input, processing, and output.
* `json.loads` → Converts the model’s JSON-formatted response into a Python dictionary.
* Returns a dictionary with `"status"` and `"response"` containing the SQL query or clarification/error message.


In [74]:
import json
def get_sql_query(genai_client, prompt, user_query):

  contents = f"""
  {employee_prompt}
  Here's the user query in english you need to work on:
  {user_query}
  """
  response = genai_client.models.generate_content(model='gemini-2.5-flash', contents=contents)
  usage_metadata = response.usage_metadata

  print(f"Input Token Count: {usage_metadata.prompt_token_count}")
  print(f"Thoughts Token Count: {response.usage_metadata.thoughts_token_count}")
  print(f"Output Token Count: {usage_metadata.candidates_token_count}")
  print(f"Total Token Count: {usage_metadata.total_token_count}")

  output = json.loads(response.text.replace('```json', '').replace('```', ''))

  return output

##**Execute SQLite Query**:

This function runs a **SQL query** on the `employee.db` database and returns the results as a **pandas DataFrame**.

**Explanation:**

* `sqlite3.connect` → Opens a connection to the SQLite database.
* `cursor.execute(query)` → Runs the SQL query.
* `cursor.description` → Retrieves column names from the query result.
* `pd.DataFrame` → Converts query results into a structured table for easy analysis.
* The connection is **safely closed** in the `finally` block to prevent resource leaks.


In [92]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='employee.db'):
  conn = None
  try:
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    print(f"\n Executing query on '{db_name}':\n {query}")

    cursor.execute(query)

    columns = [description[0] for description in cursor.description]

    results = cursor.fetchall()
    results_as_dict = [dict(zip(columns, row)) for row in results]
    results_df = pd.DataFrame(results_as_dict)

    print("Query executed successfully.")
    return results_df

  except sqlite3.Error as e:
    print(f"Database error executing query: {e}")
    return None

  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    return None

  finally :
     if conn:
      conn.close()

## **Convert Natural Language to SQL and Execute**:

This function takes a **plain English query**, uses the **GenAI model** to generate a SQL query, executes it on the `employee.db` database, and returns the results.

**Explanation:**

* `get_sql_query` → Converts the user’s natural language query into a valid SQLite query.
* `output['status']` → Checks if the query generation was successful.
* `execute_query` → Runs the generated SQL query and returns results as a pandas DataFrame.
* If the SQL cannot be generated, the function returns the **GenAI output** containing clarification or error messages.


In [76]:
def text2sql(genai_client, prompt, user_query):
  output = get_sql_query(genai_client, employee_prompt, user_query)
  if output['status'] == 'success':
    results = execute_query(output['response'])
    return results
  return output

###**Query Examples** :

In [94]:
text2sql(genai_client, employee_prompt, "Show me the order employee by country")

Input Token Count: 1748
Thoughts Token Count: 274
Output Token Count: 40
Total Token Count: 2062

 Executing query on 'employee.db':
 SELECT employee_id, first_name, last_name, country FROM Employee ORDER BY country;
Query executed successfully.


,employee_id,first_name,last_name,country
0,112,Moyra,Dagg,1
1,366,Alys,Vamplew,1
2,384,Barbee,Comettoi,10
3,403,Daron,Jankin,10
4,241,Vivyan,Lennox,103
...,...,...,...,...
995,418,Alta,Coggan,992
996,130,Inglebert,Wildber,996
997,369,Francesco,Purshouse,996
998,292,Christiane,Vassall,998


In [96]:
text2sql(genai_client, employee_prompt, "List employees whose salary is greater than 50000.")

Input Token Count: 1755
Thoughts Token Count: 461
Output Token Count: 34
Total Token Count: 2250

 Executing query on 'employee.db':
 SELECT * FROM Employee WHERE salary > 50000;
Query executed successfully.


,employee_id,first_name,last_name,age,email,gender,job_title,department,salary,hire_date,...,address,city,state,postal_code,country,emergency_contact_name,emergency_contact_phone,emergency_contact_relationship,manager_id,start_date
0,1,Ramon,Mc Elory,33,rmc0@cbsnews.com,514,648,Research and Development,76003.40,3/8/2021,...,1 Lerdahl Pass,Pescara,Abruzzi,65129,48,Ramon Mc Elory,393-443-5512,parent,1,8/11/2016
1,3,Philly,Huggan,47,phuggan2@pinterest.com,67,921,Training,53892.46,3/15/2022,...,0231 Stephen Road,Pescara,Abruzzi,65129,756,Philly Huggan,385-683-8587,parent,3,7/3/2020
2,5,Mady,McPhate,24,mmcphate4@twitter.com,142,662,Legal,105506.07,4/3/2014,...,57 Harper Center,Pescara,Abruzzi,65129,397,Mady McPhate,981-762-2757,sibling,5,4/14/2021
3,7,Verile,Sea,58,vsea6@phpbb.com,10,26,Engineering,137660.52,10/21/2018,...,91 Tomscot Way,Pescara,Abruzzi,65129,855,Verile Sea,145-940-6451,friend,7,1/28/2015
4,8,Adella,Guitel,22,aguitel7@earthlink.net,59,949,Research and Development,68701.97,6/25/2021,...,102 Service Street,Pescara,Abruzzi,65129,892,Adella Guitel,744-288-8287,sibling,8,4/27/2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,993,Conchita,Garlic,25,cgarlicrk@guardian.co.uk,45,857,Product Management,81757.07,2/1/2018,...,421 Summerview Alley,Pescara,Abruzzi,65129,705,Conchita Garlic,130-968-1868,spouse,993,3/29/2013
843,994,Court,Laing,26,claingrl@google.de,399,664,Marketing,134957.91,11/29/2011,...,9100 Bellgrove Court,Pescara,Abruzzi,65129,907,Court Laing,750-593-0384,spouse,994,3/8/2010
844,996,Beitris,Dougill,40,bdougillrn@cbsnews.com,613,774,Engineering,80997.74,2/27/2010,...,87575 Sullivan Crossing,Pescara,Abruzzi,65129,280,Beitris Dougill,329-314-5575,spouse,996,3/25/2019
845,999,Danell,Haugen,26,dhaugenrq@list-manage.com,161,313,Marketing,56662.78,6/27/2022,...,96812 Grover Lane,Pescara,Abruzzi,65129,333,Danell Haugen,196-689-7735,friend,999,12/25/2015


In [99]:
text2sql(genai_client, employee_prompt, "Show employees ordered by last_name ascending.")

Input Token Count: 1750
Thoughts Token Count: 107
Output Token Count: 31
Total Token Count: 1888

 Executing query on 'employee.db':
 SELECT * FROM Employee ORDER BY last_name ASC;
Query executed successfully.


,employee_id,first_name,last_name,age,email,gender,job_title,department,salary,hire_date,...,address,city,state,postal_code,country,emergency_contact_name,emergency_contact_phone,emergency_contact_relationship,manager_id,start_date
0,310,Marni,Abbot,64,mabbot8l@cargocollective.com,966,315,Sales,49265.11,4/4/2010,...,31580 Dovetail Crossing,Pescara,Abruzzi,65129,223,Marni Abbot,968-183-9068,parent,310,8/10/2018
1,395,Buckie,Abramowsky,60,babramowskyay@eepurl.com,543,279,Accounting,92176.06,1/2/2015,...,50 Larry Point,Pescara,Abruzzi,65129,741,Buckie Abramowsky,393-178-6805,sibling,395,11/4/2020
2,432,Guglielmo,Adacot,65,gadacotbz@rakuten.co.jp,572,207,Sales,116981.60,11/20/2013,...,95 Aberg Hill,Pescara,Abruzzi,65129,316,Guglielmo Adacot,612-889-6612,friend,432,9/3/2014
3,698,Papagena,Addington,59,paddingtonjd@unblog.fr,332,924,Training,47404.12,12/17/2010,...,6 Pierstorff Circle,Pescara,Abruzzi,65129,353,Papagena Addington,406-100-6541,spouse,698,4/7/2021
4,519,Carolina,Alday,59,caldayee@networkadvertising.org,260,457,Engineering,87390.42,5/10/2017,...,1 Bayside Trail,Pescara,Abruzzi,65129,821,Carolina Alday,993-498-5180,sibling,519,8/8/2016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,343,Audra,Yarnold,19,ayarnold9i@cbsnews.com,93,129,Marketing,32545.07,6/19/2012,...,4211 Lotheville Street,Pescara,Abruzzi,65129,267,Audra Yarnold,357-145-5699,parent,343,3/4/2015
996,673,Trish,Yellowlea,25,tyellowleaio@loc.gov,500,459,Human Resources,61952.54,7/4/2011,...,95 Prairieview Place,Pescara,Abruzzi,65129,988,Trish Yellowlea,368-711-2713,spouse,673,4/6/2021
997,836,Hanna,Yukhnov,20,hyukhnovn7@wp.com,163,735,Support,135677.09,7/23/2013,...,36784 Kings Parkway,Pescara,Abruzzi,65129,953,Hanna Yukhnov,936-867-3058,spouse,836,5/11/2012
998,256,Carmella,Zupo,59,czupo73@narod.ru,597,451,Marketing,89132.83,1/2/2013,...,236 Brickson Park Street,Pescara,Abruzzi,65129,725,Carmella Zupo,565-316-1285,friend,256,8/4/2022


In [100]:
text2sql(genai_client, employee_prompt, "Find the average salary of all employees.")

Input Token Count: 1749
Thoughts Token Count: 164
Output Token Count: 28
Total Token Count: 1941

 Executing query on 'employee.db':
 SELECT AVG(salary) FROM Employee;
Query executed successfully.


,AVG(salary)
0,90721.67332


In [101]:
text2sql(genai_client, employee_prompt, "Get the total number of employees in each department.")

Input Token Count: 1751
Thoughts Token Count: 63
Output Token Count: 39
Total Token Count: 1853

 Executing query on 'employee.db':
 SELECT department, COUNT(employee_id) AS total_employees FROM Employee GROUP BY department;
Query executed successfully.


,department,total_employees
0,Accounting,83
1,Business Development,74
2,Engineering,108
3,Human Resources,89
4,Legal,87
5,Marketing,81
6,Product Management,72
7,Research and Development,78
8,Sales,83
9,Services,77


# **Summary**:

This setup allows you to **query the employee database using plain English**. It works in three steps:

1. **Convert Natural Language to SQL** – The `get_sql_query` function uses Google’s GenAI model to translate a user question into a valid SQLite query.
2. **Execute SQL** – The `execute_query` function runs the generated query on the `employee.db` database and returns results in a **pandas DataFrame**.
3. **Integrate** – The `text2sql` function ties everything together: it converts the user query, checks for success, executes the SQL, and returns the results or clarifications/errors.

This approach allows **non-technical users** to interact with the database easily, without writing SQL manually.
